# 06d fine-tuning follow-ups: learning rate, linear probe, sampling rate, seed ensemble, calibration

Step 6 extension. Everything here starts from the same `pretrain/best` and is compared with the reference arm of 06b (H12, last-token pooling, seeds 0 to 2) trained on this machine with the same code, so no number is compared across machines. Nothing here needs new model code; the hybrid head (6a's "isolated large amounts") is notebook 06e.

**What is trained, and why.** Fine-tuning in this project is full-parameter: all 11.06M weights of the pretrained body keep training at a tenth of the pretraining learning rate, plus a new 384 x 2 head. Each experiment changes exactly one thing against that recipe:

| experiment | parameters trained | what changes | why |
|---|---|---|---|
| reference (06b H12) | all 11,062,656 | nothing: lr 5e-5, 3 epochs, batch 64, cosine with 5% warmup, best checkpoint by val PR-AUC, normals sampled to 5% | the guide's recipe |
| 1 learning rate | all | lr 1e-5, 2e-5, 1e-4, 2e-4, one seed each, selected on val; the winner gets three seeds | the XGBoost baseline had a 12-point grid, the transformer had none; the comparison is otherwise asymmetric |
| 2 linear probe | only the head (768) | body frozen; pretrained body vs random body; lr 1e-3 | how much the pretrained representation is worth on its own, without retraining the body |
| 3 sampling rate | all | neg_keep_rate 0.02 and 0.10 (0.05 is the reference), three seeds each | the one imbalance number the guide fixed, tested for ranking and for calibration |
| 4 seed ensemble | none | average of the three reference seeds' test scores | row-level disagreement between seeds; what production would deploy |
| 5 calibration | none | raw vs sampling-corrected vs temperature-scaled scores, T fitted on val | whether the score can be read as a probability |

Every training cell prints the actual trainable-parameter count before it starts (Stage 6d-1 shows the three shapes side by side), so the table above is checked by the run itself. Resumable, finished runs skipped by json; rerun from the top after an interruption. Budget on Apple MPS: 4 + 2 + 6 + 6 = up to 18 fine-tuning runs, about 7 hours (probes are much faster); on a T4 about 3 hours, but the reference would then have to be retrained there.

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## Stage 6d-0: data, the H12 examples, and the reference arm from 06b

In [ ]:
import json, time, os, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from pathlib import Path
from tfm.tokenizer import TransactionTokenizer
from tfm.finetune import build_examples
from tfm.metrics import evaluate_scores, mean_sd, correct_for_sampling
from tfm.ablate import finetune_run, run_name, run_paths, best_checkpoint, score_test, trainable_summary, freeze_body, load_model
from tfm.extras import LRS, KEEPS, REF_LR, REF_KEEP, PROBE_LR, spec, overrides_for, select_lr
from tfm.runtime import accelerator

FT_OVERRIDES = {}                                   # empty in a real run; the smoke run sets tiny values here
P, PRE, RES = f"{ROOT}/data/processed", f"{ROOT}/pretrain/best", f"{ROOT}/results"
os.makedirs(RES, exist_ok=True)
df = pd.read_parquet(f"{P}/transactions.parquet")
tok = TransactionTokenizer.load(f"{PRE}/tokenizer.json")
ids_all = tok.encode(df)
card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
PAD, BOS, V = tok.vocab["<PAD>"], tok.vocab["<BOS>"], len(tok.vocab)
ALERT, SEEDS = 0.006, (0, 1, 2)
KEYS = ["pr_auc", "roc_auc", "recall_at_alert", "card_discovery", "fp_ratio"]
DEV = accelerator()
examples = {("val", 12): build_examples(ids_all, card, split, fraud, "val", 12, PAD, BOS),
            ("test", 12): build_examples(ids_all, card, split, fraud, "test", 12, PAD, BOS)}
Xv, yv, iv = examples[("val", 12)]; Xt, yt, it = examples[("test", 12)]

def read_json(name):
    for p in (f"{RES}/{name}.json", f"results/metrics/{name}.json"):
        if os.path.exists(p): return json.load(open(p))
    raise FileNotFoundError(name)
REF = [read_json(f"06b_H12_last_seed{k}") for k in SEEDS]
REF_SCORES = [np.load(f"{ROOT}/ablate/H12_last/seed{k}/test_scores.npy") for k in SEEDS]
ref = mean_sd([r["test"] for r in REF]); REF_SD = ref["pr_auc"]["sd"]
here = describe()
same = all(r["runtime"].get("machine") == here["machine"] and r["runtime"].get("accelerator") == here["accelerator"] for r in REF)
print("accelerator:", DEV, here)
print(f"reference 06b H12 last, {len(REF)} seeds, trained on {REF[0]['runtime']}: test PR-AUC {ref['pr_auc']['mean']:.4f} +- {REF_SD:.4f}")
print("same machine as the reference:", same if same else "NO: every comparison below would be cross-machine; stop and retrain the reference here")

## Stage 6d-1: what "fine-tuning" trains here, counted on the real model

Three shapes used below, built from `pretrain/best` and counted part by part. Full fine-tuning trains everything, including the token and position embeddings; the linear probe trains the head only.

In [ ]:
rows = []
for label, init, freeze in (("full fine-tuning (reference, lr sweep, sampling)", "pretrained", False), ("linear probe, pretrained body", "pretrained", True), ("linear probe, random body", "scratch", True)):
    m = load_model(init, "last", PRE, V, PAD, BOS)
    if freeze: freeze_body(m)
    rows.append({"setting": label, **trainable_summary(m)})
    del m
shapes = pd.DataFrame(rows).set_index("setting")
display(shapes.style.format("{:,}"))
print("learning rates: reference", REF_LR, "| sweep", LRS, "| probe", PROBE_LR, "   sampling: reference", REF_KEEP, "| sweep", KEEPS)

## Stage 6d-2: learning-rate sweep

One seed per rate, chosen on validation PR-AUC exactly as the XGBoost grid was; the reference 5e-5 competes with its own seed-0 validation score. Test numbers are shown for information only; selection never reads them. If a rate other than 5e-5 wins, it gets seeds 1 and 2 so it can be compared with the reference as a 3-seed mean.

In [ ]:
t0 = time.time()
sweep = [finetune_run(spec("lr", seed=0, lr=lr), ids_all, card, split, fraud, PAD, BOS, V, PRE, ROOT, RES, examples, keep=REF_KEEP, alert=ALERT,
                      overrides={**overrides_for(spec("lr", seed=0, lr=lr)), **FT_OVERRIDES}) for lr in LRS]
ref0 = REF[0]
lr_tab = pd.DataFrame([{"lr": s["lr"], "seed": 0, "best step": s["best_step"], "val PR-AUC": s["val_at_best"]["pr_auc"], "test PR-AUC": s["test"]["pr_auc"], "test recall": s["test"]["recall_at_alert"], "min": s["wall_min"]} for s in sweep]
                      + [{"lr": REF_LR, "seed": 0, "best step": ref0["best_step"], "val PR-AUC": ref0["val_at_best"]["pr_auc"], "test PR-AUC": ref0["test"]["pr_auc"], "test recall": ref0["test"]["recall_at_alert"], "min": ref0["wall_min"]}]).sort_values("lr")
display(lr_tab.style.format({"lr": "{:.0e}", "val PR-AUC": "{:.4f}", "test PR-AUC": "{:.4f}", "test recall": "{:.4f}"}).hide(axis="index"))
LR_SEL = select_lr(sweep, REF_LR, ref0["val_at_best"]["pr_auc"])
print(f"selected on val: lr {LR_SEL:.0e}" + ("  (the reference; nothing more to train)" if LR_SEL == REF_LR else "  -> training seeds 1 and 2 at this rate"))
lr_runs = [s for s in sweep if s["lr"] == LR_SEL]
if LR_SEL != REF_LR:
    lr_runs += [finetune_run(spec("lr", seed=k, lr=LR_SEL), ids_all, card, split, fraud, PAD, BOS, V, PRE, ROOT, RES, examples, keep=REF_KEEP, alert=ALERT,
                             overrides={**overrides_for(spec("lr", seed=k, lr=LR_SEL)), **FT_OVERRIDES}) for k in (1, 2)]
lr_sel = mean_sd([s["test"] for s in lr_runs]) if LR_SEL != REF_LR else ref
print(f"lr stage {(time.time() - t0) / 60:.1f} min this session")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(lr_tab["lr"], lr_tab["val PR-AUC"], "o-", label="val PR-AUC (selection)"); ax.plot(lr_tab["lr"], lr_tab["test PR-AUC"], "s--", label="test PR-AUC (information)")
ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel("PR-AUC, seed 0"); ax.axvline(REF_LR, c="gray", ls=":", label="reference 5e-5"); ax.legend(); ax.set_title("learning-rate sweep")
plt.tight_layout(); fig.savefig(f"{RES}/06d_lr.png", dpi=120); plt.show()

## Stage 6d-3: linear probe

Body frozen, only the 384 x 2 head trains (lr 1e-3, same epochs and sampling). Two bodies: the pretrained one and a random one of the same shape. The gap between the two probes is the value of the representation alone; the gap between the pretrained probe and full fine-tuning is what retraining the body adds on top.

In [ ]:
t0 = time.time()
probe = {init: [finetune_run(spec("probe", seed=k, init=init), ids_all, card, split, fraud, PAD, BOS, V, PRE, ROOT, RES, examples, keep=REF_KEEP, alert=ALERT,
                             overrides={**overrides_for(spec("probe", seed=k, init=init)), **FT_OVERRIDES}) for k in SEEDS] for init in ("pretrained", "scratch")}
probe_agg = {init: mean_sd([s["test"] for s in runs]) for init, runs in probe.items()}
ptab = pd.DataFrame([{"arm": "probe, pretrained body (head only)", "trainable": probe["pretrained"][0]["trainable_params"], **{k: f"{probe_agg['pretrained'][k]['mean']:.4f} +- {probe_agg['pretrained'][k]['sd']:.4f}" for k in KEYS}},
                     {"arm": "probe, random body (head only)", "trainable": probe["scratch"][0]["trainable_params"], **{k: f"{probe_agg['scratch'][k]['mean']:.4f} +- {probe_agg['scratch'][k]['sd']:.4f}" for k in KEYS}},
                     {"arm": "full fine-tuning (reference)", "trainable": REF[0].get("trainable_params", "all"), **{k: f"{ref[k]['mean']:.4f} +- {ref[k]['sd']:.4f}" for k in KEYS}}])
display(ptab.style.hide(axis="index"))
print(f"representation alone (pretrained probe - random probe): {probe_agg['pretrained']['pr_auc']['mean'] - probe_agg['scratch']['pr_auc']['mean']:+.4f} PR-AUC;  "
      f"retraining the body adds (full - pretrained probe): {ref['pr_auc']['mean'] - probe_agg['pretrained']['pr_auc']['mean']:+.4f}   |  {(time.time() - t0) / 60:.1f} min this session")

## Stage 6d-4: sampling rate of the normals

The guide fixes "keep all frauds, sample the normals to 5%". Here 2% and 10%, three seeds each, full fine-tuning at the reference rate. Fewer normals mean a faster run and a more inflated raw probability; more normals the opposite. Ranking metrics and the sampling-corrected mean probability are both reported.

In [ ]:
t0 = time.time()
keep_runs = {kp: [finetune_run(spec("keep", seed=k, keep=kp), ids_all, card, split, fraud, PAD, BOS, V, PRE, ROOT, RES, examples, keep=kp, alert=ALERT,
                               overrides={**overrides_for(spec("keep", seed=k, keep=kp)), **FT_OVERRIDES}) for k in SEEDS] for kp in KEEPS}
keep_agg = {kp: mean_sd([s["test"] for s in runs]) for kp, runs in keep_runs.items()}
true_rate = float(yt.mean())
rows = []
for kp, runs, agg in [(REF_KEEP, REF, ref)] + [(kp, keep_runs[kp], keep_agg[kp]) for kp in KEEPS]:
    rows.append({"neg_keep_rate": kp, "train examples": int(np.mean([s["n_train_examples"] for s in runs])), "PR-AUC": f"{agg['pr_auc']['mean']:.4f} +- {agg['pr_auc']['sd']:.4f}",
                 "recall@0.6%": f"{agg['recall_at_alert']['mean']:.4f}", "FP ratio": f"{agg['fp_ratio']['mean']:.3f}",
                 "mean corrected prob": f"{np.mean([s['mean_prob_test_corrected'] for s in runs]):.4f}", "true fraud rate": f"{true_rate:.4f}", "min per run": round(float(np.mean([s["wall_min"] for s in runs])), 1)})
display(pd.DataFrame(rows).sort_values("neg_keep_rate").style.hide(axis="index"))
print(f"sampling stage {(time.time() - t0) / 60:.1f} min this session")

## Stage 6d-5: seed ensemble

No training. The three reference seeds' test scores are averaged. First the per-seed numbers and how much they disagree row by row, then the ensemble.

In [ ]:
S = np.stack(REF_SCORES, 1)
ens = S.mean(1)
per_seed = [evaluate_scores(yt, S[:, k], card[it], ALERT) for k in range(S.shape[1])]
ens_m = evaluate_scores(yt, ens, card[it], ALERT)
def flagged(score): f = np.zeros(len(score), bool); f[np.argsort(-score, kind="stable")[:max(1, int(round(ALERT * len(score))))]] = True; return f
F = np.stack([flagged(S[:, k]) for k in range(S.shape[1])], 1); fe = flagged(ens); fr = yt == 1
spread = S.max(1) - S.min(1)
etab = pd.DataFrame([{"model": f"seed {k}", **{m: per_seed[k][m] for m in KEYS}} for k in range(S.shape[1])] + [{"model": "ensemble (mean of 3)", **{m: ens_m[m] for m in KEYS}}])
display(etab.style.format({m: "{:.4f}" for m in KEYS}).hide(axis="index"))
ensemble = {"per_seed": per_seed, "ensemble": ens_m, "rows_spread_over_0.5": int((spread > 0.5).sum()), "rows_spread_over_0.1": int((spread > 0.1).sum()),
            "frauds_caught_all_seeds": int((F[fr].all(1)).sum()), "frauds_caught_any_seed": int((F[fr].any(1)).sum()), "frauds_caught_ensemble": int(fe[fr].sum()), "n_fraud": int(fr.sum())}
print(f"rows where the three seeds' scores spread by more than 0.5: {ensemble['rows_spread_over_0.5']:,} of {len(yt):,} (more than 0.1: {ensemble['rows_spread_over_0.1']:,})")
print(f"frauds flagged at {ALERT:.1%}: by all three seeds {ensemble['frauds_caught_all_seeds']:,}, by at least one {ensemble['frauds_caught_any_seed']:,}, by the ensemble {ensemble['frauds_caught_ensemble']:,}, of {ensemble['n_fraud']:,}")

## Stage 6d-6: calibration

Seed 0's best checkpoint re-scores the validation examples; a temperature T is fitted there on the sampling-corrected log-odds and applied to the test scores. Three versions of the same test scores are compared: raw, corrected for the 5% sampling (`metrics.correct_for_sampling`), and corrected + temperature. Ranking does not change (PR-AUC is printed to show it); only the probability reading does.

In [ ]:
from transformers import GPT2ForSequenceClassification
from tfm.calibrate import reliability_table, ece, fit_temperature, apply_temperature
ck = best_checkpoint(f"{ROOT}/ablate/H12_last/seed0")
m = GPT2ForSequenceClassification.from_pretrained(ck); m.config.pad_token_id = PAD
t0 = time.time(); p_val = score_test(m, Xv, PAD, DEV); del m
print(f"val re-scored from {Path(ck).relative_to(ROOT)} in {time.time() - t0:.0f} s")
p_test = REF_SCORES[0]
versions = {"raw": (p_val, p_test), "sampling-corrected": (correct_for_sampling(p_val, REF_KEEP), correct_for_sampling(p_test, REF_KEEP))}
T = fit_temperature(yv, versions["sampling-corrected"][0])
versions["corrected + temperature"] = (apply_temperature(versions["sampling-corrected"][0], T), apply_temperature(versions["sampling-corrected"][1], T))
calib = {"temperature": T, "true_rate_test": float(yt.mean()), "versions": {}}
rows = []
for name, (pv, pt) in versions.items():
    r = {"version": name, "ECE test": ece(yt, pt), "ECE val": ece(yv, pv), "mean pred test": float(pt.mean()), "true rate test": float(yt.mean()), "PR-AUC test": evaluate_scores(yt, pt, card[it], ALERT)["pr_auc"]}
    rows.append(r); calib["versions"][name] = {k: v for k, v in r.items() if k != "version"}; calib["versions"][name]["reliability_test"] = reliability_table(yt, pt).to_dict(orient="records")
display(pd.DataFrame(rows).style.format({c: "{:.4f}" for c in rows[0] if c != "version"}).hide(axis="index"))
print(f"fitted temperature T = {T:.3f} (T > 1 means the corrected scores were still overconfident)")
fig, ax = plt.subplots(figsize=(5.2, 4.6))
for name, (pv, pt) in versions.items():
    t = reliability_table(yt, pt); ax.plot(t["mean_pred"], t["obs_rate"], "o-", label=name)
ax.plot([0, 1], [0, 1], c="gray", ls=":", label="perfect"); ax.set_xlabel("mean predicted probability (bin)"); ax.set_ylabel("observed fraud rate (bin)"); ax.set_title("reliability, test rows, seed 0"); ax.legend()
plt.tight_layout(); fig.savefig(f"{RES}/06d_calibration.png", dpi=120); plt.show()

## Stage 6d-7: everything against the reference, in units of the reference's seed sd

In [ ]:
def row(arm, agg, note=""):
    d = agg["pr_auc"]["mean"] - ref["pr_auc"]["mean"]
    return {"arm": arm, "n seeds": agg["pr_auc"]["n_seeds"], **{k: f"{agg[k]['mean']:.4f} +- {agg[k]['sd']:.4f}" for k in KEYS}, "PR-AUC - ref": d, "in ref sd": d / max(REF_SD, 1e-9), "note": note}
summary = [row("reference: full fine-tuning, lr 5e-5, keep 0.05 (06b H12)", ref)]
summary.append(row(f"lr selected on val: {LR_SEL:.0e}", lr_sel, "same as reference" if LR_SEL == REF_LR else "3 seeds"))
summary.append(row("linear probe, pretrained body", probe_agg["pretrained"], "head only"))
summary.append(row("linear probe, random body", probe_agg["scratch"], "head only"))
for kp in KEEPS: summary.append(row(f"neg_keep_rate {kp:.2f}", keep_agg[kp]))
summary.append({"arm": "seed ensemble (mean of 3)", "n seeds": 3, **{k: f"{ens_m[k]:.4f}" for k in KEYS}, "PR-AUC - ref": ens_m["pr_auc"] - ref["pr_auc"]["mean"], "in ref sd": (ens_m["pr_auc"] - ref["pr_auc"]["mean"]) / max(REF_SD, 1e-9), "note": "one model, no sd"})
stab = pd.DataFrame(summary)
display(stab.style.format({"PR-AUC - ref": "{:+.4f}", "in ref sd": "{:+.1f}"}).hide(axis="index"))
out = {"reference": {"runs": [r["name"] for r in REF], "aggregate": ref, "runtime": REF[0]["runtime"]}, "shapes": shapes.to_dict(orient="index"),
       "lr_sweep": sweep, "lr_selected": LR_SEL, "lr_selected_aggregate": lr_sel, "probe": {k: {"runs": v, "aggregate": probe_agg[k]} for k, v in probe.items()},
       "keep": {str(k): {"runs": v, "aggregate": keep_agg[k]} for k, v in keep_runs.items()}, "ensemble": ensemble, "calibration": calib,
       "summary": stab.to_dict(orient="records"), "runtime": describe()}
with open(f"{RES}/06d_extras.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved 06d_extras.json:", list(out))

## What to read off the numbers

- Learning rate: if the selected rate is 5e-5, the asymmetry with the tuned XGBoost is closed and the step-5 comparison stands as is; if another rate wins, its 3-seed mean replaces the reference in the report's headline and the gain is stated as what tuning bought.
- Linear probe: the pretrained-probe minus random-probe gap is the value of the representation with nothing else changed; the full minus pretrained-probe gap is what retraining the body adds. If the pretrained probe alone already beats XGBoost, the representation is the story; if it does not, the body's retraining is.
- Sampling rate: a tie across 2 / 5 / 10 percent says the ranking does not depend on the sampling choice and only the probability does (see the corrected mean against the true rate).
- Ensemble: row-level spread between seeds is the number a validator quotes; the ensemble's count of frauds flagged against "all three" and "at least one" says what averaging buys at the same alert budget.
- Calibration: ECE after correction and after temperature; if T is close to 1 the analytic correction was enough. PR-AUC is identical across the three versions by construction.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git pull --rebase -q && git push -q   # the other 06c group may have pushed meanwhile; its files are disjoint
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

In [ ]:
if on_colab():
    drive.flush_and_unmount()